# 04 — Error Handling & Exceptions


## 1. Exception Hierarchy

In [ ]:
# Visualize exception hierarchy
def print_hierarchy(cls, indent=0):
    print(' ' * indent + cls.__name__)
    for sub in cls.__subclasses__():
        print_hierarchy(sub, indent + 2)

print_hierarchy(BaseException)

## 2. try/except/else/finally

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('Cannot divide by zero')
        return None
    except TypeError as e:
        print(f'Type error: {e}')
        return None
    else:
        print(f'Success: {a}/{b} = {result}')
        return result
    finally:
        print('finally always runs')

safe_divide(10, 2)
print()
safe_divide(10, 0)
print()
safe_divide(10, 'x')

## 3. Custom Exceptions

In [ ]:
class AppError(Exception):
    """Base application exception."""

class ValidationError(AppError):
    def __init__(self, field: str, message: str):
        self.field = field
        self.message = message
        super().__init__(f"Validation failed for '{field}': {message}")

class NotFoundError(AppError):
    def __init__(self, resource: str, id_: int):
        super().__init__(f"{resource} with id={id_} not found")
        self.resource = resource
        self.id = id_

class DatabaseError(AppError):
    pass

# Test
errors = [
    ValidationError('email', 'invalid format'),
    NotFoundError('User', 42),
    DatabaseError('Connection refused'),
]

for err in errors:
    try:
        raise err
    except ValidationError as e:
        print(f'Validation: field={e.field}, msg={e.message}')
    except NotFoundError as e:
        print(f'NotFound: {e.resource} id={e.id}')
    except AppError as e:
        print(f'AppError: {e}')

## 4. Exception Chaining

In [ ]:
# raise X from Y — explicit chaining
def load_config(path):
    try:
        with open(path) as f:
            import json
            return json.load(f)
    except FileNotFoundError as e:
        raise RuntimeError(f'Config file not found: {path}') from e
    except json.JSONDecodeError as e:
        raise ValueError(f'Invalid JSON in config: {path}') from e

try:
    load_config('/nonexistent/config.json')
except RuntimeError as e:
    print(f'Error: {e}')
    print(f'Caused by: {e.__cause__}')

## 5. contextlib.suppress

In [ ]:
from contextlib import suppress
import os

# Without suppress
try:
    os.remove('/tmp/nonexistent_file_xyz.txt')
except FileNotFoundError:
    pass

# With suppress — cleaner
with suppress(FileNotFoundError):
    os.remove('/tmp/nonexistent_file_xyz.txt')

print('No error raised')

## 6. Retry Pattern

In [ ]:
import time
import random
from functools import wraps

def retry(max_attempts=3, delay=0.1, exceptions=(Exception,)):
    """Decorator: retry on specified exceptions."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    print(f'Attempt {attempt}/{max_attempts} failed: {e}')
                    if attempt < max_attempts:
                        time.sleep(delay * attempt)  # exponential backoff
            raise last_exc
        return wrapper
    return decorator

call_count = 0

@retry(max_attempts=3, delay=0.01)
def flaky_operation():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f'Connection failed (attempt {call_count})')
    return 'Success!'

result = flaky_operation()
print(f'Result: {result}')

## 7. Exception Groups (Python 3.11+)

In [ ]:
import sys

if sys.version_info >= (3, 11):
    try:
        raise ExceptionGroup('validation errors', [
            ValueError('name is required'),
            TypeError('age must be int'),
            ValueError('email is invalid'),
        ])
    except* ValueError as eg:
        print(f'ValueErrors: {[str(e) for e in eg.exceptions]}')
    except* TypeError as eg:
        print(f'TypeErrors: {[str(e) for e in eg.exceptions]}')
else:
    print('ExceptionGroup requires Python 3.11+')